In [22]:
from datasets import load_dataset, Dataset
from transformers import AutoModelForSeq2SeqLM
from transformers import AutoTokenizer
from transformers import GenerationConfig
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments, BitsAndBytesConfig
import pandas as pd
import numpy as np
from peft import get_peft_model, LoraConfig, TaskType
import torch
import os

In [23]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [24]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [25]:
torch.cuda.empty_cache()

In [26]:
train = pd.read_pickle('train_clean.pkl')
#validation = pd.read_pickle('validation_clean.pkl')

In [27]:
train_frag = train.sample(n=100, random_state=42)

In [28]:
len_summary = train_frag['summary'].apply(len)
len_summary.mean()

np.float64(1239.03)

In [29]:
len_summary.std()

np.float64(350.42114503678124)

In [30]:
ds_train = Dataset.from_pandas(train_frag)

In [31]:
ds_train

Dataset({
    features: ['text', 'summary', 'doc_id'],
    num_rows: 100
})

In [32]:
bnb_config = BitsAndBytesConfig(load_in_8bit=True)
model_name = 'google/flan-t5-large'
model = AutoModelForSeq2SeqLM.from_pretrained(model_name, quantization_config=bnb_config, device_map="auto").to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading weights: 100%|██████████| 558/558 [00:01<00:00, 315.65it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [33]:
tokenizer.pad_token = tokenizer.eos_token

In [34]:
token_lengths = [
    len(tokenizer(summary)["input_ids"])
    for summary in train["summary"]
]

truncated_pct = np.mean(np.array(token_lengths) > 512)

print(f"{truncated_pct}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (810 > 512). Running this sequence through the model will result in indexing errors


0.01


In [35]:
input_tokens = [
    len(tokenizer(text)["input_ids"])
    for text in train["text"].sample(min(500, len(train)))
]

print(pd.Series(input_tokens).describe())

count     500.000000
mean      512.606000
std        79.425459
min       414.000000
25%       466.000000
50%       491.500000
75%       528.000000
max      1056.000000
dtype: float64


In [36]:
summary_tokens = [
    len(tokenizer(summary)["input_ids"])
    for summary in train["summary"].sample(min(500, len(train)))
]

print(pd.Series(summary_tokens).describe())

count    500.000000
mean     254.452000
std       90.017075
min       20.000000
25%      197.000000
50%      246.000000
75%      301.000000
max      929.000000
dtype: float64


In [37]:
max_len = 1024

In [38]:
def tokenize_function(x):
    texto = [f"Summarize the following text: {texto}" for texto in x['text']]
    inputs = tokenizer(texto, max_length=max_len, truncation=True)
    labels = tokenizer(x['summary'], max_length=512, truncation=True)

    inputs['labels'] = labels['input_ids']
    return inputs

In [39]:
ds_tokenized = ds_train.map(tokenize_function, batched=True, remove_columns=ds_train.column_names, batch_size=8)

Map: 100%|██████████| 100/100 [00:00<00:00, 1508.62 examples/s]


In [40]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,  # para flan-t5
    r=16, #32,                              # rango de LoRA
    lora_alpha=32, #64,
    lora_dropout=1e-5, #0.05,
    target_modules=["q", "v", "k", "o"]         # capas a adaptar
)

In [41]:
model = get_peft_model(model, lora_config)

In [42]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100  # ignora el padding en el loss
)

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./model_testing/flan-t5-finetuned",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    logging_steps=50,
    predict_with_generate=True,       # importante para seq2seq
    #fp16=True                         # si tienes GPU compatible
)

In [44]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=ds_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator  # aquí
)

In [45]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 1}.
c:\Users\User\anaconda3\envs\summarization\Lib\site-packages\bitsandbytes\autograd\_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss
50,18.266553


TrainOutput(global_step=70, training_loss=18.16245596749442, metrics={'train_runtime': 19396.644, 'train_samples_per_second': 0.052, 'train_steps_per_second': 0.004, 'total_flos': 2512482983239680.0, 'train_loss': 18.16245596749442, 'epoch': 10.0})

In [47]:
print(ds_tokenized[0])

{'input_ids': [12198, 1635, 1737, 8, 826, 1499, 10, 13200, 19, 46, 1832, 6275, 12, 8, 1057, 13, 3, 5096, 4992, 138, 18, 52, 88, 3600, 1489, 1693, 5, 21917, 2296, 19612, 7, 2123, 1388, 12, 3, 20424, 4358, 3, 5096, 4992, 138, 3149, 13, 3, 17, 5167, 11, 1108, 3703, 16, 4290, 930, 11, 3, 21275, 124, 21, 529, 18, 9842, 1208, 14877, 5, 37, 6565, 138, 1693, 9127, 7, 8, 569, 11, 3, 24703, 2530, 7, 5147, 383, 6565, 11, 5510, 10281, 7, 139, 2930, 22739, 1693, 11, 5023, 1681, 1693, 6, 3, 5, 5945, 60, 1693, 13009, 186, 1832, 5559, 1414, 76, 6, 1673, 3, 5, 86, 4656, 12, 8, 119, 386, 46, 9, 14991, 2097, 6, 5945, 60, 1693, 25375, 8, 11041, 433, 13, 1499, 3239, 5, 938, 3, 12072, 11, 3, 20424, 8, 1646, 8641, 11, 119, 3, 24703, 6803, 13, 3, 9, 1499, 6, 5349, 1693, 795, 3, 9, 72, 9517, 1705, 13, 8, 1309, 344, 607, 11, 1681, 5, 26230, 5054, 7, 995, 4341, 12, 11770, 70, 554, 18, 14973, 1573, 6, 378, 33, 7463, 16, 70, 793, 2625, 7, 12, 734, 42, 7280, 984, 16, 1353, 13, 8, 2530, 7, 1742, 12317, 12, 135, 3128

In [48]:
batch = data_collator([ds_tokenized[0]])

batch = {k: v.to(model.device) for k, v in batch.items()}

outputs = model(**batch)

print(outputs.loss)

c:\Users\User\anaconda3\envs\summarization\Lib\site-packages\bitsandbytes\autograd\_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


tensor(3.2511, device='cuda:0', grad_fn=<NllLossBackward0>)
